In [0]:
%sql
USE CATALOG rearc

In [0]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from urllib.parse import urljoin


In [0]:
url = "https://download.bls.gov/pub/time.series/pr/"

# BLS requests that you include an email address so they can contact you
# if there are issues with your usage. Replace the email below with yours.
bls_email = "sbatlouni@gmail.com"
headers = {
    "User-Agent": f"requests_library ({bls_email})",
}
response = requests.get(url, headers=headers)

In [0]:
response.raise_for_status()

# The page is an IIS directory listing: each file line is
# "<date> <time> <AM/PM> <size> <a href=...>name</a>" inside a <pre> block.
soup = BeautifulSoup(response.text, "html.parser")
rows = []
for a in soup.find("pre").find_all("a"):
    meta = a.previous_sibling
    if not isinstance(meta, str) or not meta.strip():
        continue  # skips "[To Parent Directory]"
    date, time, ampm, size = meta.split()
    rows.append({
        "file_name": a.get_text(),
        "last_modified": pd.to_datetime(f"{date} {time} {ampm}", format="%m/%d/%Y %I:%M %p"),
        "size_bytes": int(size),
        "url": urljoin(url, a["href"]),
    })

df = pd.DataFrame(rows)
df